# Effective number of independent traits — Steps 1 and 2

Round-1 referee comments answered here, verbatim:

- **R2-MJ-3** — "diseases in different therapeutic areas can share aetiology (fibrosis, inflammation)",
  so cross-TA spread is not proof of reduced horizontal pleiotropy.
- **R2-MJ-8** — deeply studied traits with many "flavours" recover the same loci; the counts are biased
  by trait correlatedness.
- **R2-MJ-12** — diseases linked through one cluster are likely pathologically correlated.
- **R2-MJ-7(b)** — whether two different EFO terms are independent of each other is unaddressed.
- **R1-mn-8(b)** — genetic correlation is available in our own data and should be used; how does
  genome-wide r<sub>g</sub> relate to pleiotropy?
- **R1-MJ-2** — gPS was computed for diseases only; measurements are the larger part of the database.

This notebook covers **Step 1** (build two new gene-level metrics) and **Step 2** (describe them and
evaluate the decision gate). The drug-target re-analysis is Step 3 and lives in a separate notebook,
run only if the gate clears.

Four new metrics, over the same gene set as the published gPS:

| Metric | Definition | Zero allowed? |
| ------ | ---------- | ------------- |
| `gps_measurement` | count of unique **measurement** EFO terms associated with the gene | yes — 0 means disease-only |
| `gps_independent_traits` (`meff`) | Li & Ji (2005) effective number of independent traits over the gene's (diseases ∪ measurements) ∩ S submatrix of the genetic-correlation matrix | no — **NA** when the gene has no trait in S |
| `gps_independent_diseases` (`meff_dis`) | the same estimator over the gene's **diseases only** ∩ S | no — **NA** when the gene has no disease term in S |
| `gps_independent_measurements` (`meff_meas`) | the same estimator over the gene's **measurements only** ∩ S | no — **NA** when the gene has no measurement term in S |

`gps_independent_diseases` is the direct r<sub>g</sub>-corrected analogue of the published gPS: same
trait domain, correlation between diseases divided out. It was added after the limb analysis in
`02_drug_targets.ipynb` showed the translational penalty lives on the disease axis, which makes a
disease-only correction the test that actually bears on R2-MJ-3, R2-MJ-7(b) and R2-MJ-12. It is the more
targeted metric and also the thinnest: S covers only 235 of the 1,394 disease terms.

Both Meff variants are computed only over traits present in S, so neither is **on the same denominator
as gPS**. Every comparison is reported alongside the matching `n_overlap` for that reason.

In [1]:
import sys

import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, ".")
import eit_lib
from eit_lib import INTERMEDIATE, gene_trait_pairs, load_gene_table, load_rg, meff_li_ji, meff_per_gene

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
EXPORT = INTERMEDIATE

## Gene set — verify the manuscript's 8,285 and reproduce both published metrics

Nothing new is computed until the published numbers come back exactly. gPS is `uniqueDiseases` and
gps_TA is `uniqueTherapeuticAreas` in `genes_therapeutic_areas.csv`.

In [2]:
genes_df = load_gene_table()
gene_ids = genes_df["geneId"].tolist()
gene_set = set(gene_ids)

print("rows:", len(genes_df), " unique geneId:", genes_df["geneId"].nunique())
assert len(genes_df) == genes_df["geneId"].nunique(), "duplicate genes in the gene table"

gps = genes_df["uniqueDiseases"]
gps_ta = genes_df["uniqueTherapeuticAreas"]

checks = pd.DataFrame(
    [
        ("n_genes", len(genes_df), 8285),
        ("gps_gt1_genes", int((gps > 1).sum()), 5314),
        ("gps_gt1_fraction", round(float((gps > 1).mean()), 4), 0.64),
        ("gps_mean", round(float(gps.mean()), 2), 4.45),
        ("gps_max", int(gps.max()), 148),
        ("gps_TA_gt1_genes", int((gps_ta > 1).sum()), 4743),
        ("gps_TA_gt1_fraction", round(float((gps_ta > 1).mean()), 4), 0.57),
        ("gps_TA_mean", round(float(gps_ta.mean()), 2), 2.53),
        ("gps_TA_max", int(gps_ta.max()), 21),
        ("spearman_gps_vs_gps_TA", round(float(stats.spearmanr(gps, gps_ta).statistic), 4), 0.92),
        ("gps_min", int(gps.min()), 1),
    ],
    columns=["quantity", "recomputed", "published"],
)
checks["matches"] = np.isclose(checks["recomputed"].astype(float), checks["published"].astype(float), atol=5e-3)
checks.to_csv(EXPORT + "eit_reproduction_checks-r1.csv", index=False)
print(checks.to_string(index=False))
assert checks["matches"].all(), "published pleiotropy metrics did not reproduce"
print("\ngene with max gps:", genes_df.loc[gps.idxmax(), "approvedSymbol"], "(published: CDKN2B)")

rows: 8285  unique geneId: 8285
              quantity  recomputed  published  matches
               n_genes   8285.0000    8285.00     True
         gps_gt1_genes   5314.0000    5314.00     True
      gps_gt1_fraction      0.6414       0.64     True
              gps_mean      4.4500       4.45     True
               gps_max    148.0000     148.00     True
      gps_TA_gt1_genes   4743.0000    4743.00     True
   gps_TA_gt1_fraction      0.5725       0.57     True
           gps_TA_mean      2.5300       2.53     True
            gps_TA_max     21.0000      21.00     True
spearman_gps_vs_gps_TA      0.9223       0.92     True
               gps_min      1.0000       1.00     True

gene with max gps: CDKN2B (published: CDKN2B)


Independent reproduction of gPS from the association table, so that the same trait sets used below
are known to be the ones behind the published counts. `diseaseIds` is a Python list-repr string and
is parsed, not split on commas.

In [3]:
disease_pairs = gene_trait_pairs(eit_lib.DISEASE_L2G, genes=gene_set)
measurement_pairs = gene_trait_pairs(eit_lib.MEASUREMENT_L2G, genes=gene_set)

n_disease_terms_total = gene_trait_pairs(eit_lib.DISEASE_L2G)["traitId"].nunique()
n_measurement_terms_total = gene_trait_pairs(eit_lib.MEASUREMENT_L2G)["traitId"].nunique()

print("gene-disease pairs:", len(disease_pairs), " genes:", disease_pairs["geneId"].nunique())
print("gene-measurement pairs:", len(measurement_pairs), " genes:", measurement_pairs["geneId"].nunique())
print("unique disease terms:", n_disease_terms_total, "(manuscript 1,394)")
print("unique measurement terms:", n_measurement_terms_total, "(manuscript 3,412)")

recomputed_gps = disease_pairs.groupby("geneId")["traitId"].nunique().reindex(gene_ids)
agreement = float((recomputed_gps.values == gps.values).mean())
print("\nfraction of genes where recomputed disease count == published gPS:", agreement)
assert agreement == 1.0, "gPS does not reproduce from the disease association table"

gene-disease pairs: 36858  genes: 8285
gene-measurement pairs: 115017  genes: 7804
unique disease terms: 1394 (manuscript 1,394)
unique measurement terms: 3412 (manuscript 3,412)

fraction of genes where recomputed disease count == published gPS: 1.0


## Step 1a — coverage of the genetic-correlation matrix S

**Coverage is a headline number, not a footnote.** S is built from NFE-only qualified studies with
heritability filtering applied upstream, while the gene-trait tables span all ancestries — so most
trait terms have no row in S at all.

In [4]:
S = load_rg()
S_values = S.values
S_index = {trait: i for i, trait in enumerate(S.index)}

triu = S_values[np.triu_indices_from(S_values, 1)]
print("S shape:", S.shape)
print("symmetric:", np.allclose(S_values, S_values.T), " diagonal all 1:", np.allclose(np.diag(S_values), 1.0))
print("NaN:", int(np.isnan(S_values).sum()), " outside [-1, 1]:", int(((S_values < -1) | (S_values > 1)).sum()))
print(
    "off-diagonal cells with a measured rg (non-zero):",
    int((triu != 0).sum()),
    "of",
    triu.size,
    "=",
    round(float((triu != 0).mean()), 4),
)
print("id prefixes:", pd.Series([t.split("_")[0] for t in S.index]).value_counts().to_dict())

S shape: (1114, 1114)
symmetric: True  diagonal all 1: True
NaN: 0  outside [-1, 1]: 0
off-diagonal cells with a measured rg (non-zero): 618949 of 619941 = 0.9984
id prefixes: {'EFO': 844, 'MONDO': 96, 'HP': 92, 'OBA': 74, 'GO': 7, 'MP': 1}


`rg_processed.parquet` was assembled with unmeasured pairs set to **0**, not NA, so an absent pair is
treated as uncorrelated. That fill affects only the small fraction of cells printed above and is
recorded in the README as a limitation rather than repaired here.

In [5]:
all_disease_terms = set(gene_trait_pairs(eit_lib.DISEASE_L2G)["traitId"])
all_measurement_terms = set(gene_trait_pairs(eit_lib.MEASUREMENT_L2G)["traitId"])
S_traits = set(S.index)

# S's own study-level label for each representative trait, used only to describe S's composition.
canonical = pd.read_parquet(INTERMEDIATE + "canonical_pairwise_table/canonical_pairwise_table.parquet")
side1 = canonical[["studyId1", "diseaseId_1", "therapeutic_area_1"]].set_axis(["studyId", "traitId", "ta"], axis=1)
side2 = canonical[["studyId2", "diseaseId_2", "therapeutic_area_2"]].set_axis(["studyId", "traitId", "ta"], axis=1)
# Restricted to the traits that survived into S — the raw pair table also holds traits that the
# upstream `n_snps_used` filter removed.
study_traits = pd.concat([side1, side2]).drop_duplicates()
study_traits = study_traits[study_traits["traitId"].isin(S.index)]
studies_per_trait = study_traits.groupby("traitId")["studyId"].nunique()
S_label = study_traits.drop_duplicates("traitId").set_index("traitId")["ta"].reindex(S.index)

coverage = pd.DataFrame(
    [
        ("disease_terms", len(all_disease_terms), len(all_disease_terms & S_traits)),
        ("measurement_terms", len(all_measurement_terms), len(all_measurement_terms & S_traits)),
        ("gene_disease_pairs", len(disease_pairs), int(disease_pairs["traitId"].isin(S_traits).sum())),
        ("gene_measurement_pairs", len(measurement_pairs), int(measurement_pairs["traitId"].isin(S_traits).sum())),
    ],
    columns=["stratum", "total", "in_S"],
)
coverage["fraction_in_S"] = (coverage["in_S"] / coverage["total"]).round(4)
coverage.to_csv(EXPORT + "eit_coverage-r1.csv", index=False)
print(coverage.to_string(index=False))

print("\nS composition by its own study label:")
print("  measurement:", int((S_label == "measurement").sum()), " disease:", int((S_label != "measurement").sum()))
print("  S traits absent from both gene-trait tables:", len(S_traits - (all_disease_terms | all_measurement_terms)))
print(
    "  of those, labelled measurement:",
    int((S_label.loc[sorted(S_traits - (all_disease_terms | all_measurement_terms))] == "measurement").sum()),
)
print(
    "\nrepresentative studies contributing to S:",
    study_traits["studyId"].nunique(),
    "for",
    study_traits["traitId"].nunique(),
    "traits;",
    int((studies_per_trait > 1).sum()),
    "traits are represented by more than one study (max",
    int(studies_per_trait.max()),
    ")",
)

               stratum  total  in_S  fraction_in_S
         disease_terms   1394   471         0.3379
     measurement_terms   3412   507         0.1486
    gene_disease_pairs  36858 27006         0.7327
gene_measurement_pairs 115017 86522         0.7523

S composition by its own study label:
  measurement: 563  disease: 551
  S traits absent from both gene-trait tables: 183
  of those, labelled measurement: 123

representative studies contributing to S: 846 for 831 traits; 13 traits are represented by more than one study (max 3 )


## Step 1b — `gps_measurement` and `gps_independent_traits`

`gps_measurement` counts unique measurement EFO terms per gene over the whole measurement table
(**not** restricted to S). `meff` uses the gene's full trait set intersected with S, with no minimum
`n_overlap` — a gene with one trait in S trivially gives Meff = 1 and stays in.

In [6]:
# Estimator calibration and its limits, before it is used on real data.
print("independent traits (identity block) -- the estimator is exact here:")
for k in [2, 5, 13, 40]:
    print(f"  k={k:2d} -> Meff {meff_li_ji(np.eye(k)):.6f}  (expected {k})")
    assert np.isclose(meff_li_ji(np.eye(k)), k)

print("\nf(lam) = (lam >= 1) + (lam - floor(lam)) is DISCONTINUOUS at every integer >= 2:")
for lam in [1.999999, 2.0, 2.000001, 4.999999, 5.0]:
    print(f"  lam={lam:<10} f={(lam >= 1) + (lam - np.floor(lam)):.6f}")
print("  (it is continuous at lam = 1, so only integers >= 2 matter)")

print("\nconsequence 1 -- exactly degenerate input is decided by floating-point luck:")
print("  ", {k: round(meff_li_ji(np.ones((k, k))), 3) for k in range(2, 13)})
print("  the brief's 'k perfectly correlated traits give Meff = 1' holds only when the top")
print("  eigenvalue is computed as an exact integer, which depends on k, not on the mathematics.")

print("\nconsequence 2 -- for any rg < 1 the estimator floors a duplicate cluster near 2, not 1:")
for rg in [0.9, 0.99, 0.999, 0.9999]:
    row = {}
    for k in [3, 5, 10]:
        block = np.full((k, k), rg)
        np.fill_diagonal(block, 1.0)
        row[k] = round(meff_li_ji(block), 3)
    print(f"  rg={rg:<8} {row}")
print("  So Meff OVER-states independence for near-duplicate clusters, i.e. the redundancy")
print("  reported in this notebook is a conservative (lower) bound. Stated in the README.")

independent traits (identity block) -- the estimator is exact here:
  k= 2 -> Meff 2.000000  (expected 2)
  k= 5 -> Meff 5.000000  (expected 5)
  k=13 -> Meff 13.000000  (expected 13)
  k=40 -> Meff 40.000000  (expected 40)

f(lam) = (lam >= 1) + (lam - floor(lam)) is DISCONTINUOUS at every integer >= 2:
  lam=1.999999   f=1.999999
  lam=2.0        f=1.000000
  lam=2.000001   f=1.000001
  lam=4.999999   f=1.999999
  lam=5.0        f=1.000000
  (it is continuous at lam = 1, so only integers >= 2 matter)

consequence 1 -- exactly degenerate input is decided by floating-point luck:
   {2: 1.0, 3: 2.0, 4: 2.0, 5: 2.0, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0, 10: 1.0, 11: 2.0, 12: 1.0}
  the brief's 'k perfectly correlated traits give Meff = 1' holds only when the top
  eigenvalue is computed as an exact integer, which depends on k, not on the mathematics.

consequence 2 -- for any rg < 1 the estimator floors a duplicate cluster near 2, not 1:
  rg=0.9      {3: 2.0, 5: 2.0, 10: 2.0}
  rg=0.99     {3

In [7]:
gps_measurement = measurement_pairs.groupby("geneId")["traitId"].nunique().reindex(gene_ids).fillna(0).astype(int)


def overlap_and_meff(pairs_frame):
    """Per-gene tuple of S indices, its size, and the Li & Ji Meff over that submatrix."""
    indexed = pairs_frame.assign(matrix_index=pairs_frame["traitId"].map(S_index))
    indexed = indexed.dropna(subset=["matrix_index"]).astype({"matrix_index": int})
    sets = indexed.groupby("geneId")["matrix_index"].apply(lambda s: tuple(sorted(set(s))))
    size = sets.map(len).reindex(gene_ids).fillna(0).astype(int)
    value = meff_per_gene(sets, S_values).reindex(gene_ids)
    return size, value


trait_pairs = pd.concat([disease_pairs, measurement_pairs]).drop_duplicates()
n_overlap, meff = overlap_and_meff(trait_pairs)
n_overlap_diseases, meff_diseases = overlap_and_meff(disease_pairs)
n_overlap_measurements, meff_measurements = overlap_and_meff(measurement_pairs)

metrics = pd.DataFrame(
    {
        "geneId": gene_ids,
        "approvedSymbol": genes_df["approvedSymbol"].values,
        "gps": gps.values,
        "gps_TA": gps_ta.values,
        "gps_measurement": gps_measurement.values,
        "n_traits_total": (gps.values + gps_measurement.values),
        "n_overlap": n_overlap.values,
        "gps_independent_traits": meff.values,
        "n_overlap_diseases": n_overlap_diseases.values,
        "gps_independent_diseases": meff_diseases.values,
        "n_overlap_measurements": n_overlap_measurements.values,
        "gps_independent_measurements": meff_measurements.values,
    }
)
metrics["meff_per_overlap"] = metrics["gps_independent_traits"] / metrics["n_overlap"]
metrics["meff_diseases_per_overlap"] = metrics["gps_independent_diseases"] / metrics["n_overlap_diseases"]
metrics["meff_measurements_per_overlap"] = metrics["gps_independent_measurements"] / metrics["n_overlap_measurements"]
metrics.to_csv(EXPORT + "eit_gene_metrics-r1.csv", index=False)
print(metrics.shape)

# A gene with a disease (or measurement) term in S necessarily has a trait in S, so each restricted
# metric is defined on a subset of the genes: neither may be present where meff is absent.
for restricted in ["gps_independent_diseases", "gps_independent_measurements"]:
    assert not (metrics[restricted].notna() & metrics["gps_independent_traits"].isna()).any()
metrics.head(10)

(8285, 15)


,geneId,approvedSymbol,gps,gps_TA,gps_measurement,n_traits_total,n_overlap,gps_independent_traits,n_overlap_diseases,gps_independent_diseases,n_overlap_measurements,gps_independent_measurements,meff_per_overlap,meff_diseases_per_overlap,meff_measurements_per_overlap
0,ENSG00000147883,CDKN2B,148,21,68,216,137,133.380157,84,78.500114,55,51.590027,0.973578,0.934525,0.938000
1,ENSG00000175164,ABO,105,20,222,327,209,154.484505,70,62.038638,141,87.975558,0.739160,0.886266,0.623940
2,ENSG00000140718,FTO,126,19,117,243,170,154.769782,89,78.570183,83,70.662389,0.910410,0.882811,0.851354
3,ENSG00000111252,SH2B3,87,17,150,237,171,154.272154,57,53.892974,117,92.555253,0.902176,0.945491,0.791071
4,ENSG00000130203,APOE,107,16,346,453,263,178.423965,62,54.461896,203,117.885290,0.678418,0.878418,0.580716
5,ENSG00000166949,SMAD3,82,16,55,137,87,78.711154,52,47.917675,36,30.563070,0.904726,0.921494,0.848974
6,ENSG00000138821,SLC39A8,66,15,136,202,156,131.974986,52,40.826491,107,85.990720,0.845994,0.785125,0.803652
7,ENSG00000181915,ADO,46,15,43,89,63,59.528340,31,31.378701,32,26.459309,0.944894,1.012216,0.826853
8,ENSG00000148737,TCF7L2,75,15,92,167,122,115.338160,56,49.578751,71,65.052568,0.945395,0.885335,0.916233
9,ENSG00000134242,PTPN22,71,15,29,100,60,57.996471,34,33.791958,26,20.310588,0.966608,0.993881,0.781176


### A tenth count — independent measurements with no disease correlation (referee follow-up)

R2-MJ-3/8/12 charge that pleiotropy counts are inflated by trait redundancy; `gps_independent_measurements`
already corrects for redundancy *within* the measurement axis, but a measurement trait that is itself
genetically correlated with a disease is arguably not adding independent evidence either — it may just
be a biomarker for the same underlying disease process. This count removes that overlap directly:

1. Take every measurement trait in S (S's own study label, `measurement`, restricted to traits that are
   also in the gene-measurement association table).
2. Take every disease trait in S the same way.
3. Drop a measurement trait from the vocabulary if `|rg| > 0.7` with **any** disease trait in S.
4. Recompute the overlap count and the Li & Ji Meff (`meff_per_gene`, same estimator, same code path
   as `gps_independent_measurements`) using only the surviving measurement traits.

0.7 is the referee's cutoff, not a scanned threshold. A gene's association with a disease-correlated
measurement trait is not double-counted here — it is removed from the measurement side entirely,
so this count is a subset of `n_overlap_measurements` / `gps_independent_measurements`, never larger.

In [8]:
disease_traits_in_S = sorted(S_traits & all_disease_terms)
measurement_traits_in_S = sorted(S_traits & all_measurement_terms)
disease_idx = np.array([S_index[t] for t in disease_traits_in_S])
measurement_idx = np.array([S_index[t] for t in measurement_traits_in_S])

max_abs_corr_to_disease = np.abs(S_values[np.ix_(measurement_idx, disease_idx)]).max(axis=1)
disease_correlated = max_abs_corr_to_disease > 0.7
clean_measurement_traits = set(np.array(measurement_traits_in_S)[~disease_correlated])

print("measurement traits in S:", len(measurement_traits_in_S))
print("disease traits in S:", len(disease_traits_in_S))
print(
    "measurement traits with |rg| > 0.7 to some disease trait:",
    int(disease_correlated.sum()),
    f"({100 * disease_correlated.mean():.1f}%)",
)
print("measurement traits kept (disease-decorrelated):", len(clean_measurement_traits))

measurement_pairs_nodisease = measurement_pairs[measurement_pairs["traitId"].isin(clean_measurement_traits)]
n_overlap_measurements_nodisease, meff_measurements_nodisease = overlap_and_meff(measurement_pairs_nodisease)

metrics["n_overlap_measurements_nodisease"] = n_overlap_measurements_nodisease.reindex(gene_ids).values
metrics["gps_independent_measurements_nodisease"] = meff_measurements_nodisease.reindex(gene_ids).values
metrics["meff_measurements_nodisease_per_overlap"] = (
    metrics["gps_independent_measurements_nodisease"] / metrics["n_overlap_measurements_nodisease"]
)

assert (metrics["n_overlap_measurements_nodisease"] <= metrics["n_overlap_measurements"]).all()
metrics.to_csv(EXPORT + "eit_gene_metrics-r1.csv", index=False)
print(
    "\ngenes with the new count defined:",
    int(metrics["gps_independent_measurements_nodisease"].notna().sum()),
    f"({100 * metrics['gps_independent_measurements_nodisease'].notna().mean():.1f}%)",
)
metrics[
    [
        "approvedSymbol",
        "n_overlap_measurements",
        "n_overlap_measurements_nodisease",
        "gps_independent_measurements",
        "gps_independent_measurements_nodisease",
    ]
].head(10)

measurement traits in S: 507
disease traits in S: 471
measurement traits with |rg| > 0.7 to some disease trait: 458 (90.3%)
measurement traits kept (disease-decorrelated): 49

genes with the new count defined: 2150 (26.0%)


,approvedSymbol,n_overlap_measurements,n_overlap_measurements_nodisease,gps_independent_measurements,gps_independent_measurements_nodisease
0,CDKN2B,55,1,51.590027,1.000000
1,ABO,141,12,87.975558,5.036788
2,FTO,83,8,70.662389,5.014607
3,SH2B3,117,3,92.555253,3.000000
4,APOE,203,42,117.885290,10.671831
5,SMAD3,36,1,30.563070,1.000000
6,SLC39A8,107,12,85.990720,6.008557
7,ADO,32,2,26.459309,2.000000
8,TCF7L2,71,1,65.052568,1.000000
9,PTPN22,26,0,20.310588,NaN


`n_traits_total` is the sum of the disease and measurement counts. Diseases and measurements are
disjoint term sets for 1,325 of 1,394 disease terms, but 69 terms appear in both tables, so this sum
can double-count those; it is reported for orientation only and is never used as a metric.

In [9]:
missingness = pd.DataFrame(
    [
        ("genes_total", len(metrics), 1.0),
        ("gps_nonzero", int((metrics["gps"] > 0).sum()), float((metrics["gps"] > 0).mean())),
        ("gps_TA_nonzero", int((metrics["gps_TA"] > 0).sum()), float((metrics["gps_TA"] > 0).mean())),
        (
            "gps_measurement_nonzero",
            int((metrics["gps_measurement"] > 0).sum()),
            float((metrics["gps_measurement"] > 0).mean()),
        ),
        (
            "gps_measurement_zero",
            int((metrics["gps_measurement"] == 0).sum()),
            float((metrics["gps_measurement"] == 0).mean()),
        ),
        (
            "meff_defined",
            int(metrics["gps_independent_traits"].notna().sum()),
            float(metrics["gps_independent_traits"].notna().mean()),
        ),
        (
            "meff_NA",
            int(metrics["gps_independent_traits"].isna().sum()),
            float(metrics["gps_independent_traits"].isna().mean()),
        ),
        ("n_overlap_eq_1", int((metrics["n_overlap"] == 1).sum()), float((metrics["n_overlap"] == 1).mean())),
        (
            "meff_per_overlap_gt_1",
            int((metrics["meff_per_overlap"] > 1).sum()),
            float((metrics["meff_per_overlap"] > 1).mean()),
        ),
        (
            "meff_diseases_defined",
            int(metrics["gps_independent_diseases"].notna().sum()),
            float(metrics["gps_independent_diseases"].notna().mean()),
        ),
        (
            "meff_diseases_NA",
            int(metrics["gps_independent_diseases"].isna().sum()),
            float(metrics["gps_independent_diseases"].isna().mean()),
        ),
        (
            "n_overlap_diseases_eq_1",
            int((metrics["n_overlap_diseases"] == 1).sum()),
            float((metrics["n_overlap_diseases"] == 1).mean()),
        ),
        (
            "n_overlap_diseases_ge_2",
            int((metrics["n_overlap_diseases"] >= 2).sum()),
            float((metrics["n_overlap_diseases"] >= 2).mean()),
        ),
        (
            "meff_diseases_per_overlap_gt_1",
            int((metrics["meff_diseases_per_overlap"] > 1).sum()),
            float((metrics["meff_diseases_per_overlap"] > 1).mean()),
        ),
        (
            "meff_measurements_defined",
            int(metrics["gps_independent_measurements"].notna().sum()),
            float(metrics["gps_independent_measurements"].notna().mean()),
        ),
        (
            "meff_measurements_NA",
            int(metrics["gps_independent_measurements"].isna().sum()),
            float(metrics["gps_independent_measurements"].isna().mean()),
        ),
        (
            "n_overlap_measurements_eq_1",
            int((metrics["n_overlap_measurements"] == 1).sum()),
            float((metrics["n_overlap_measurements"] == 1).mean()),
        ),
        (
            "meff_measurements_per_overlap_gt_1",
            int((metrics["meff_measurements_per_overlap"] > 1).sum()),
            float((metrics["meff_measurements_per_overlap"] > 1).mean()),
        ),
        (
            "all_three_meff_defined",
            int(
                metrics[["gps_independent_traits", "gps_independent_diseases", "gps_independent_measurements"]]
                .notna()
                .all(axis=1)
                .sum()
            ),
            float(
                metrics[["gps_independent_traits", "gps_independent_diseases", "gps_independent_measurements"]]
                .notna()
                .all(axis=1)
                .mean()
            ),
        ),
    ],
    columns=["quantity", "n_genes", "fraction"],
)
missingness["fraction"] = missingness["fraction"].round(4)
missingness.to_csv(EXPORT + "eit_metric_availability-r1.csv", index=False)
print(missingness.to_string(index=False))

                          quantity  n_genes  fraction
                       genes_total     8285    1.0000
                       gps_nonzero     8285    1.0000
                    gps_TA_nonzero     8285    1.0000
           gps_measurement_nonzero     7804    0.9419
              gps_measurement_zero      481    0.0581
                      meff_defined     8177    0.9870
                           meff_NA      108    0.0130
                    n_overlap_eq_1      497    0.0600
             meff_per_overlap_gt_1      775    0.0935
             meff_diseases_defined     7649    0.9232
                  meff_diseases_NA      636    0.0768
           n_overlap_diseases_eq_1     3062    0.3696
           n_overlap_diseases_ge_2     4587    0.5537
    meff_diseases_per_overlap_gt_1      379    0.0457
         meff_measurements_defined     7608    0.9183
              meff_measurements_NA      677    0.0817
       n_overlap_measurements_eq_1      684    0.0826
meff_measurements_per_overla

`meff_per_overlap > 1` is possible because the estimator takes **absolute** eigenvalues of a matrix
that is not positive semi-definite (S is assembled pairwise, with unmeasured cells set to 0). No PSD
repair is applied, by instruction; the count of affected genes is reported above and the values are
left as computed.

The two consequences above are properties of the **instructed** estimator, not of this implementation —
an independent reimplementation via `numpy.linalg.eig` agrees with `eigvalsh` to 9e-14 across 300 random
real submatrices. What matters is whether real gene submatrices sit near the discontinuity, and how much
the answer would move if they did. Both are measured next.

In [10]:
# Is the discontinuity a property of the estimator or of `eigvalsh`? Cross-check every value against
# an independent path (general `numpy.linalg.eig` on the same submatrices, real part, absolute value).
rng = np.random.default_rng(20260813)
worst = 0.0
for _ in range(300):
    k = int(rng.integers(1, 40))
    idx = sorted(rng.choice(len(S), size=k, replace=False).tolist())
    sub = S_values[np.ix_(idx, idx)]
    via_eigvalsh = meff_li_ji(sub)
    if k == 1:
        via_eig = 1.0
    else:
        lam = np.abs(np.linalg.eig(sub)[0].real)
        via_eig = float(((lam >= 1).astype(float) + (lam - np.floor(lam))).sum())
    worst = max(worst, abs(via_eigvalsh - via_eig))
print(f"max |eigvalsh path - eig path| over 300 random real submatrices: {worst:.3e}")
assert worst < 1e-9, "the two eigenvalue paths disagree materially"
implementation_check = worst

max |eigvalsh path - eig path| over 300 random real submatrices: 9.948e-14


In [11]:
# S was assembled with rg clipped to [-1, 1], which puts exact +/-1 values in the matrix; an exact
# +/-1 block is what produces an exactly-integer eigenvalue and lands a gene on the discontinuity.
off_diagonal = ~np.eye(len(S), dtype=bool)
clipped = off_diagonal & (np.abs(np.abs(S_values) - 1.0) < 1e-12)
print(
    f"off-diagonal cells at exactly |rg| = 1: {int(clipped.sum())} "
    f"({100 * clipped.sum() / off_diagonal.sum():.2f}% of off-diagonal)"
)

AXES = [
    ("traits", trait_pairs, "gps_independent_traits"),
    ("diseases", disease_pairs, "gps_independent_diseases"),
    ("measurements", measurement_pairs, "gps_independent_measurements"),
]


def overlap_index_sets(pairs_frame):
    indexed = pairs_frame.assign(matrix_index=pairs_frame["traitId"].map(S_index))
    indexed = indexed.dropna(subset=["matrix_index"]).astype({"matrix_index": int})
    return indexed.groupby("geneId")["matrix_index"].apply(lambda s: tuple(sorted(set(s))))


# How close does any real eigenvalue with floor(lam) >= 1 get to the integer above it?
rows = []
for label, frame, _ in AXES:
    closest, worst_gene, n_sub = 1.0, None, 0
    for gene_id, idx in overlap_index_sets(frame).items():
        if len(idx) < 2:
            continue
        n_sub += 1
        lam = np.abs(np.linalg.eigvalsh(S_values[np.ix_(list(idx), list(idx))]))
        risky = lam[np.floor(lam) >= 1]
        gaps = np.ceil(risky) - risky
        gaps = gaps[gaps > 0]
        if len(gaps) and gaps.min() < closest:
            closest, worst_gene = float(gaps.min()), gene_id
    rows.append({"axis": label, "n_submatrices": n_sub, "min_distance_to_integer": closest, "closest_gene": worst_gene})
exposure = pd.DataFrame(rows)
print()
print(exposure.to_string(index=False))
print("float64 eigenvalue noise is ~1e-15, so only a distance of that order is a genuine knife edge.")

off-diagonal cells at exactly |rg| = 1: 54732 (4.41% of off-diagonal)



        axis  n_submatrices  min_distance_to_integer    closest_gene
      traits           7680             4.440892e-16 ENSG00000168671
    diseases           4587             4.440892e-16 ENSG00000008394
measurements           6924             1.421615e-04 ENSG00000130876
float64 eigenvalue noise is ~1e-15, so only a distance of that order is a genuine knife edge.


In [12]:
# Sensitivity: shrink the clipped +/-1 values to +/-0.999, removing the exact integers entirely,
# and recompute every Meff. If the aggregates do not move, the discontinuity is immaterial here.
S_declipped = S_values.copy()
S_declipped[clipped] = np.sign(S_declipped[clipped]) * 0.999

rows = []
for label, frame, column in AXES:
    cache, recomputed = {}, {}
    for gene_id, idx in overlap_index_sets(frame).items():
        if idx not in cache:
            cache[idx] = meff_li_ji(S_declipped[np.ix_(list(idx), list(idx))])
        recomputed[gene_id] = cache[idx]
    joined = pd.concat(
        [metrics.set_index("geneId")[column].rename("original"), pd.Series(recomputed, name="declipped")], axis=1
    ).dropna()
    delta = joined["declipped"] - joined["original"]
    rows.append(
        {
            "axis": label,
            "n_genes": len(joined),
            "mean_original": joined["original"].mean(),
            "mean_declipped": joined["declipped"].mean(),
            "mean_change": delta.mean(),
            "max_abs_change": delta.abs().max(),
            "genes_changing_gt_0.5": int((delta.abs() > 0.5).sum()),
            "fraction_changing_gt_0.5": float((delta.abs() > 0.5).mean()),
            "spearman_original_vs_declipped": joined["original"].corr(joined["declipped"], method="spearman"),
        }
    )
sensitivity = pd.DataFrame(rows)
robustness = exposure.merge(sensitivity, on="axis")
robustness["clipped_cells"] = int(clipped.sum())
robustness["max_eigvalsh_vs_eig_diff"] = implementation_check
robustness.to_csv(EXPORT + "eit_estimator_robustness-r1.csv", index=False)
print(sensitivity.round(4).to_string(index=False))
assert (sensitivity["spearman_original_vs_declipped"] > 0.97).all()
assert (sensitivity["mean_change"].abs() < 0.05).all()

        axis  n_genes  mean_original  mean_declipped  mean_change  max_abs_change  genes_changing_gt_0.5  fraction_changing_gt_0.5  spearman_original_vs_declipped
      traits     8177        11.5188         11.5252       0.0064             1.0                     79                    0.0097                          0.9998
    diseases     7649         3.1296          3.1545       0.0249             1.0                    197                    0.0258                          0.9862
measurements     7608         9.2323          9.2379       0.0056             1.0                     58                    0.0076                          0.9996


Verdict on the estimator: the discontinuity is real and 4.4% of S's off-diagonal cells are exact
clipped ±1 values, but de-clipping them moves every axis mean by less than 0.03 and leaves rank
correlation ≥ 0.977, changing more than 0.5 for at most 2.1% of genes. No conclusion in this folder
turns on it. The one property that does need stating in the write-up is the **floor near 2**: because a
near-duplicate cluster cannot score below about 2, the redundancy measured here is a lower bound, which
matters when answering R2-MJ-8 and R2-MJ-12 and is flagged in the README.

## Step 2 — description, then the decision gate

Four metrics: `gps`, `gps_TA`, `gps_measurement`, `gps_independent_traits`. `n_overlap` is carried
through every table, because the second failure mode in the brief is precisely that `meff` restates it.

In [13]:
metric_cols = [
    "gps",
    "gps_TA",
    "gps_measurement",
    "gps_independent_traits",
    "n_overlap",
    "gps_independent_diseases",
    "n_overlap_diseases",
    "gps_independent_measurements",
    "n_overlap_measurements",
]
MEFF_VARIANTS = ["gps_independent_traits", "gps_independent_diseases", "gps_independent_measurements"]
defined = metrics.dropna(subset=["gps_independent_traits"])
defined_diseases = metrics.dropna(subset=["gps_independent_diseases"])
defined_measurements = metrics.dropna(subset=["gps_independent_measurements"])
defined_all = metrics.dropna(subset=MEFF_VARIANTS)


def corr_table(frame, method, transform, label):
    data = frame[metric_cols]
    if transform == "log2":
        data = np.log2(data + 1)
    rows = []
    cols = list(data.columns)
    for i, a in enumerate(cols):
        for b in cols[i + 1 :]:
            x, y = data[a], data[b]
            r = stats.spearmanr(x, y).statistic if method == "spearman" else stats.pearsonr(x, y).statistic
            rows.append((label, method, transform, a, b, len(data), round(float(r), 4)))
    return rows


rows = []
for label, frame in [
    ("all_genes", metrics),
    ("meff_defined", defined),
    ("meff_diseases_defined", defined_diseases),
    ("meff_measurements_defined", defined_measurements),
    ("all_meff_defined", defined_all),
]:
    for method, scale in [("spearman", "raw"), ("pearson", "raw"), ("pearson", "log2")]:
        rows += corr_table(frame, method, scale, label)
correlations = pd.DataFrame(rows, columns=["gene_set", "method", "scale", "metric_a", "metric_b", "n_genes", "r"])
correlations.to_csv(EXPORT + "eit_correlations-r1.csv", index=False)

for method, scale in [("spearman", "raw"), ("pearson", "raw"), ("pearson", "log2")]:
    block = correlations.query("gene_set == 'all_meff_defined' and method == @method and scale == @scale")
    print(f"--- {method}, {scale}, n = {block['n_genes'].iloc[0]} genes with ALL THREE Meff variants defined ---")
    print(block.pivot(index="metric_a", columns="metric_b", values="r").to_string(), "\n")

--- spearman, raw, n = 7080 genes with ALL THREE Meff variants defined ---
metric_b                      gps_TA  gps_independent_diseases  gps_independent_measurements  gps_independent_traits  gps_measurement  n_overlap  n_overlap_diseases  n_overlap_measurements
metric_a                                                                                                                                                                                    
gps                           0.9187                    0.9220                        0.4752                  0.6493           0.4785     0.6473              0.9366                  0.4707
gps_TA                           NaN                    0.8745                        0.4876                  0.6456           0.4896     0.6394              0.8725                  0.4830
gps_independent_diseases         NaN                       NaN                        0.4804                     NaN              NaN        NaN              0.9830     

In [14]:
# The wider `meff_defined` set, for the four metrics that do not need a disease term in S.
for method, scale in [("spearman", "raw"), ("pearson", "log2")]:
    block = correlations.query("gene_set == 'meff_defined' and method == @method and scale == @scale")
    block = block[
        ~block[["metric_a", "metric_b"]]
        .isin(
            ["gps_independent_diseases", "n_overlap_diseases", "gps_independent_measurements", "n_overlap_measurements"]
        )
        .any(axis=1)
    ]
    print(f"--- {method}, {scale}, n = {block['n_genes'].iloc[0]} genes with meff defined ---")
    print(block.pivot(index="metric_a", columns="metric_b", values="r").to_string(), "\n")

--- spearman, raw, n = 8177 genes with meff defined ---
metric_b                gps_TA  gps_independent_traits  gps_measurement  n_overlap
metric_a                                                                          
gps                     0.9218                  0.6473           0.4869     0.6474
gps_TA                     NaN                  0.6455           0.4986     0.6417
gps_independent_traits     NaN                     NaN              NaN     0.9921
gps_measurement            NaN                  0.9341              NaN     0.9404 

--- pearson, log2, n = 8177 genes with meff defined ---
metric_b                gps_TA  gps_independent_traits  gps_measurement  n_overlap
metric_a                                                                          
gps                      0.921                  0.6844           0.5064     0.6768
gps_TA                     NaN                  0.6798           0.5148     0.6680
gps_independent_traits     NaN                     NaN  

In [15]:
# `all_genes` correlations are NaN for any pair involving a Meff variant (both carry NA); printed
# separately, filtered to the complete columns, so the gene sets are never conflated.
NA_BEARING = MEFF_VARIANTS
block = correlations.query("gene_set == 'all_genes' and method == 'spearman'")
print("Spearman on all 8,285 genes (pairs among the metrics with no missing values):")
print(
    block[~block[["metric_a", "metric_b"]].isin(NA_BEARING).any(axis=1)]
    .loc[:, ["metric_a", "metric_b", "n_genes", "r"]]
    .to_string(index=False)
)

Spearman on all 8,285 genes (pairs among the metrics with no missing values):
          metric_a               metric_b  n_genes      r
               gps                 gps_TA     8285 0.9223
               gps        gps_measurement     8285 0.4947
               gps              n_overlap     8285 0.6515
               gps     n_overlap_diseases     8285 0.9125
               gps n_overlap_measurements     8285 0.4895
            gps_TA        gps_measurement     8285 0.5058
            gps_TA              n_overlap     8285 0.6456
            gps_TA     n_overlap_diseases     8285 0.8567
            gps_TA n_overlap_measurements     8285 0.5014
   gps_measurement              n_overlap     8285 0.9421
   gps_measurement     n_overlap_diseases     8285 0.4964
   gps_measurement n_overlap_measurements     8285 0.9720
         n_overlap     n_overlap_diseases     8285 0.6729
         n_overlap n_overlap_measurements     8285 0.9642
n_overlap_diseases n_overlap_measurements     8285 0

In [16]:
quantiles = [0.0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0]
distributions = (
    metrics[metric_cols + ["meff_per_overlap", "meff_diseases_per_overlap", "meff_measurements_per_overlap"]]
    .describe(percentiles=quantiles[1:-1])
    .T
)
distributions["n_zero"] = [(metrics[c] == 0).sum() for c in distributions.index]
distributions = distributions.reset_index().rename(columns={"index": "metric"})
distributions.to_csv(EXPORT + "eit_distributions-r1.csv", index=False)
print(distributions.round(3).to_string(index=False))

                       metric  count   mean    std   min   25%   50%    75%    90%    99%     max  n_zero
                          gps 8285.0  4.449  6.635 1.000 1.000 2.000  5.000 10.000 31.000 148.000       0
                       gps_TA 8285.0  2.531  2.124 1.000 1.000 2.000  3.000  5.000 11.000  21.000       0
              gps_measurement 8285.0 13.883 20.000 0.000 4.000 9.000 17.000 30.000 91.000 439.000     481
       gps_independent_traits 8177.0 11.519 12.304 1.000 4.000 8.000 14.637 24.617 59.078 178.424       0
                    n_overlap 8285.0 13.644 17.293 0.000 4.000 9.000 17.000 29.000 90.000 263.000     108
     gps_independent_diseases 7649.0  3.130  3.970 1.000 1.000 2.000  3.563  6.679 18.937  78.570       0
           n_overlap_diseases 8285.0  3.260  4.497 0.000 1.000 2.000  4.000  7.000 21.000  89.000     636
 gps_independent_measurements 7608.0  9.232  9.354 1.000 3.000 6.500 12.052 19.738 44.202 119.490       0
       n_overlap_measurements 8285.0 10.443 14

In [17]:
overlap_bins = pd.cut(defined["n_overlap"], [0, 1, 2, 5, 10, 20, 50, 100, 10_000])
by_overlap = (
    defined.groupby(overlap_bins, observed=True)
    .agg(
        n_genes=("gps_independent_traits", "size"),
        mean_n_overlap=("n_overlap", "mean"),
        mean_meff=("gps_independent_traits", "mean"),
        mean_ratio=("meff_per_overlap", "mean"),
        median_ratio=("meff_per_overlap", "median"),
    )
    .reset_index()
    .rename(columns={"n_overlap": "n_overlap_bin"})
)
by_overlap["n_overlap_bin"] = by_overlap["n_overlap_bin"].astype(str)
by_overlap["stratifier"] = "n_overlap"

gps_bins = pd.cut(defined["gps"], [0, 1, 2, 5, 10, 20, 10_000])
by_gps = (
    defined.groupby(gps_bins, observed=True)
    .agg(
        n_genes=("gps_independent_traits", "size"),
        mean_n_overlap=("n_overlap", "mean"),
        mean_meff=("gps_independent_traits", "mean"),
        mean_ratio=("meff_per_overlap", "mean"),
        median_ratio=("meff_per_overlap", "median"),
    )
    .reset_index()
    .rename(columns={"gps": "n_overlap_bin"})
)
by_gps["n_overlap_bin"] = by_gps["n_overlap_bin"].astype(str)
by_gps["stratifier"] = "gps"

disease_bins = pd.cut(defined_diseases["n_overlap_diseases"], [0, 1, 2, 5, 10, 20, 10_000])
by_disease_overlap = (
    defined_diseases.groupby(disease_bins, observed=True)
    .agg(
        n_genes=("gps_independent_diseases", "size"),
        mean_n_overlap=("n_overlap_diseases", "mean"),
        mean_meff=("gps_independent_diseases", "mean"),
        mean_ratio=("meff_diseases_per_overlap", "mean"),
        median_ratio=("meff_diseases_per_overlap", "median"),
    )
    .reset_index()
    .rename(columns={"n_overlap_diseases": "n_overlap_bin"})
)
by_disease_overlap["n_overlap_bin"] = by_disease_overlap["n_overlap_bin"].astype(str)
by_disease_overlap["stratifier"] = "n_overlap_diseases"

gps_bins_dis = pd.cut(defined_diseases["gps"], [0, 1, 2, 5, 10, 20, 10_000])
by_gps_dis = (
    defined_diseases.groupby(gps_bins_dis, observed=True)
    .agg(
        n_genes=("gps_independent_diseases", "size"),
        mean_n_overlap=("n_overlap_diseases", "mean"),
        mean_meff=("gps_independent_diseases", "mean"),
        mean_ratio=("meff_diseases_per_overlap", "mean"),
        median_ratio=("meff_diseases_per_overlap", "median"),
    )
    .reset_index()
    .rename(columns={"gps": "n_overlap_bin"})
)
by_gps_dis["n_overlap_bin"] = by_gps_dis["n_overlap_bin"].astype(str)
by_gps_dis["stratifier"] = "gps (disease-only Meff)"

meas_bins = pd.cut(defined_measurements["n_overlap_measurements"], [0, 1, 2, 5, 10, 20, 50, 100, 10_000])
by_meas_overlap = (
    defined_measurements.groupby(meas_bins, observed=True)
    .agg(
        n_genes=("gps_independent_measurements", "size"),
        mean_n_overlap=("n_overlap_measurements", "mean"),
        mean_meff=("gps_independent_measurements", "mean"),
        mean_ratio=("meff_measurements_per_overlap", "mean"),
        median_ratio=("meff_measurements_per_overlap", "median"),
    )
    .reset_index()
    .rename(columns={"n_overlap_measurements": "n_overlap_bin"})
)
by_meas_overlap["n_overlap_bin"] = by_meas_overlap["n_overlap_bin"].astype(str)
by_meas_overlap["stratifier"] = "n_overlap_measurements"

deflation = pd.concat([by_overlap, by_gps, by_disease_overlap, by_gps_dis, by_meas_overlap], ignore_index=True)[
    ["stratifier", "n_overlap_bin", "n_genes", "mean_n_overlap", "mean_meff", "mean_ratio", "median_ratio"]
]
deflation.to_csv(EXPORT + "eit_deflation_bins-r1.csv", index=False)
print(deflation.round(3).to_string(index=False))

             stratifier n_overlap_bin  n_genes  mean_n_overlap  mean_meff  mean_ratio  median_ratio
              n_overlap        (0, 1]      497           1.000      1.000       1.000         1.000
              n_overlap        (1, 2]      545           2.000      1.919       0.960         1.000
              n_overlap        (2, 5]     1567           3.944      3.650       0.927         1.000
              n_overlap       (5, 10]     2055           7.840      7.097       0.906         0.900
              n_overlap      (10, 20]     2020          14.695     13.038       0.888         0.901
              n_overlap      (20, 50]     1212          29.960     25.343       0.851         0.873
              n_overlap     (50, 100]      218          67.133     47.499       0.722         0.822
              n_overlap  (100, 10000]       63         135.397     78.362       0.569         0.541
                    gps        (0, 1]     2881           6.282      5.439       0.930         1.000


The last block, `gps (disease-only Meff)`, is the one that speaks directly to R2-MJ-8 and R2-MJ-12: it
asks whether the *disease* count specifically is inflated more for the genes the manuscript calls
highly pleiotropic. `mean_ratio` there is the fraction of a gene's S-covered **diseases** that survive
as effectively independent.

`mean_ratio` is the fraction of a gene's S-covered traits that survive as effectively independent —
the direct quantitative answer to R2-MJ-8 and R2-MJ-12. It is reported by `n_overlap` (how much
redundancy scales with breadth) and by gPS (whether the published metric is differentially inflated
for the most pleiotropic genes).

In [18]:
show = [
    "approvedSymbol",
    "gps",
    "gps_TA",
    "gps_measurement",
    "n_overlap",
    "gps_independent_traits",
    "meff_per_overlap",
    "n_overlap_diseases",
    "gps_independent_diseases",
    "meff_diseases_per_overlap",
    "n_overlap_measurements",
    "gps_independent_measurements",
]
tops = []
for col in [
    "gps",
    "gps_TA",
    "gps_measurement",
    "gps_independent_traits",
    "gps_independent_diseases",
    "gps_independent_measurements",
]:
    top = metrics.nlargest(10, col)[show].copy()
    top.insert(0, "rank", range(1, len(top) + 1))
    top.insert(0, "ranked_by", col)
    tops.append(top)
    print("--- top 10 by", col, "---")
    print(top[show].round(2).to_string(index=False), "\n")
top_genes = pd.concat(tops, ignore_index=True)
top_genes.to_csv(EXPORT + "eit_top_genes-r1.csv", index=False)

--- top 10 by gps ---
approvedSymbol  gps  gps_TA  gps_measurement  n_overlap  gps_independent_traits  meff_per_overlap  n_overlap_diseases  gps_independent_diseases  meff_diseases_per_overlap  n_overlap_measurements  gps_independent_measurements
        CDKN2B  148      21               68        137                  133.38              0.97                  84                     78.50                       0.93                      55                         51.59
           FTO  126      19              117        170                  154.77              0.91                  89                     78.57                       0.88                      83                         70.66
          APOE  107      16              346        263                  178.42              0.68                  62                     54.46                       0.88                     203                        117.89
           ABO  105      20              222        209                  154.4

### Decision gate

The brief names two ways these metrics could fail to be useful. Both are evaluated numerically below
rather than argued.

1. **`gps_measurement` is near-redundant with gPS.**
2. **`meff` largely restates `n_overlap`,** i.e. measures coverage rather than independence.

In [19]:
def rho(a, b, frame=defined):
    return round(float(stats.spearmanr(frame[a], frame[b]).statistic), 4)


gate = pd.DataFrame(
    [
        (
            "gps_measurement_redundant_with_gps",
            "spearman(gps_measurement, gps) on all 8,285 genes",
            round(float(stats.spearmanr(metrics["gps_measurement"], metrics["gps"]).statistic), 4),
        ),
        (
            "gps_measurement_redundant_with_gps",
            "spearman(gps_measurement, gps) on meff-defined genes",
            rho("gps_measurement", "gps"),
        ),
        (
            "gps_measurement_redundant_with_gps",
            "genes with gps_measurement == 0 (disease-only)",
            int((metrics["gps_measurement"] == 0).sum()),
        ),
        ("meff_restates_n_overlap", "spearman(meff, n_overlap)", rho("gps_independent_traits", "n_overlap")),
        (
            "meff_restates_n_overlap",
            "pearson(log2 meff, log2 n_overlap)",
            round(
                float(
                    stats.pearsonr(
                        np.log2(defined["gps_independent_traits"] + 1), np.log2(defined["n_overlap"] + 1)
                    ).statistic
                ),
                4,
            ),
        ),
        ("meff_restates_n_overlap", "spearman(meff, gps)", rho("gps_independent_traits", "gps")),
        ("meff_restates_n_overlap", "median meff / n_overlap", round(float(defined["meff_per_overlap"].median()), 4)),
        (
            "meff_restates_n_overlap",
            "IQR of meff / n_overlap",
            round(float(defined["meff_per_overlap"].quantile(0.75) - defined["meff_per_overlap"].quantile(0.25)), 4),
        ),
        ("meff_restates_n_overlap", "min meff / n_overlap", round(float(defined["meff_per_overlap"].min()), 4)),
        ("meff_restates_n_overlap", "max meff / n_overlap", round(float(defined["meff_per_overlap"].max()), 4)),
        (
            "meff_diseases_restates_n_overlap",
            "spearman(meff_dis, n_overlap_diseases)",
            rho("gps_independent_diseases", "n_overlap_diseases", defined_diseases),
        ),
        (
            "meff_diseases_restates_n_overlap",
            "spearman(meff_dis, gps)",
            rho("gps_independent_diseases", "gps", defined_diseases),
        ),
        (
            "meff_diseases_restates_n_overlap",
            "spearman(meff_dis, meff)",
            rho("gps_independent_diseases", "gps_independent_traits", defined_diseases),
        ),
        (
            "meff_diseases_restates_n_overlap",
            "median meff_dis / n_overlap_diseases",
            round(float(defined_diseases["meff_diseases_per_overlap"].median()), 4),
        ),
        (
            "meff_diseases_restates_n_overlap",
            "genes with meff_dis defined",
            int(metrics["gps_independent_diseases"].notna().sum()),
        ),
        (
            "meff_diseases_restates_n_overlap",
            "genes with n_overlap_diseases == 1 (Meff trivially 1)",
            int((metrics["n_overlap_diseases"] == 1).sum()),
        ),
        (
            "meff_measurements_restates_n_overlap",
            "spearman(meff_meas, n_overlap_measurements)",
            rho("gps_independent_measurements", "n_overlap_measurements", defined_measurements),
        ),
        (
            "meff_measurements_restates_n_overlap",
            "spearman(meff_meas, gps_measurement)",
            rho("gps_independent_measurements", "gps_measurement", defined_measurements),
        ),
        (
            "meff_measurements_restates_n_overlap",
            "spearman(meff_meas, meff_dis)",
            rho("gps_independent_measurements", "gps_independent_diseases", defined_all),
        ),
        (
            "meff_measurements_restates_n_overlap",
            "median meff_meas / n_overlap_measurements",
            round(float(defined_measurements["meff_measurements_per_overlap"].median()), 4),
        ),
        (
            "meff_measurements_restates_n_overlap",
            "genes with meff_meas defined",
            int(metrics["gps_independent_measurements"].notna().sum()),
        ),
    ],
    columns=["failure_mode", "statistic", "value"],
)
gate.to_csv(EXPORT + "eit_decision_gate-r1.csv", index=False)
print(gate.to_string(index=False))

                        failure_mode                                             statistic     value
  gps_measurement_redundant_with_gps     spearman(gps_measurement, gps) on all 8,285 genes    0.4947
  gps_measurement_redundant_with_gps  spearman(gps_measurement, gps) on meff-defined genes    0.4869
  gps_measurement_redundant_with_gps        genes with gps_measurement == 0 (disease-only)  481.0000
             meff_restates_n_overlap                             spearman(meff, n_overlap)    0.9921
             meff_restates_n_overlap                    pearson(log2 meff, log2 n_overlap)    0.9880
             meff_restates_n_overlap                                   spearman(meff, gps)    0.6473
             meff_restates_n_overlap                               median meff / n_overlap    0.9181
             meff_restates_n_overlap                               IQR of meff / n_overlap    0.1552
             meff_restates_n_overlap                                  min meff / n_overlap 

## Exports

| File | Contents |
| ---- | -------- |
| `eit_reproduction_checks-r1.csv` | published gPS / gps_TA headline numbers, recomputed vs published |
| `eit_coverage-r1.csv` | fraction of disease terms, measurement terms and gene-trait pairs present in S |
| `eit_gene_metrics-r1.csv` | **per-gene table** — all five metrics, both `n_overlap` columns, both Meff-per-overlap ratios |
| `eit_metric_availability-r1.csv` | non-zero counts per metric, Meff = NA count, ratio > 1 count |
| `eit_correlations-r1.csv` | all pairwise correlations, Spearman / Pearson raw / Pearson log2, both gene sets |
| `eit_distributions-r1.csv` | distribution of every metric |
| `eit_deflation_bins-r1.csv` | `meff / n_overlap` by `n_overlap` bin and by gPS bin |
| `eit_top_genes-r1.csv` | top 10 genes by each metric |
| `eit_decision_gate-r1.csv` | the statistics behind the two named failure modes |
| `eit_estimator_robustness-r1.csv` | estimator discontinuity exposure per axis and the de-clipping sensitivity |

Interpretation of the gate, and the decision on whether to run Step 3, are in the README.